# Analyze noise threshold

This notebook calculates an average spike triggered wavelet spectrogram for each spike type.

## Initialization - don't change

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from workflows.noise_threshold_analysis import (
    collect_aggregated_data,
    collect_pooled_data
)


from workflows.run_util.io import (
    ResultPaths,
    collect_paths_from_table,
    get_cell_info_list,
    collect_processed_paths_for_infos,
    CellWrapper
)
from param_classes.workflows.wavelet_analysis import (
    SpikeTriggeredWaveletWorkflowParams
)
from workflows.triggered_wavelet_analysis import run_analysis

params = SpikeTriggeredWaveletWorkflowParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [2]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff',
        'invitro': '#00f1feff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [3]:
params.data.raw_data.path='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [4]:
params.data.spike_features.path = 'E:\\Data\\BogiSejtek\\20251118\\Results'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [5]:
params.input = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251118\\data_invivo.xlsx',
    # 'select_rules': []
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [6]:
params.output.path = 'E:\\Data\\BogiSejtek\\20251118\\NoiseAnalysis\\invivo'

## Analysis parameters

In [7]:
threshold_cell = 'B32_d2t1c2'

In [8]:
cell_prop_table = pd.read_excel(params.input['table_path'])
cell_infos = get_cell_info_list(params.input)
cell_infos = [info for info in cell_infos if info.name != 'B6_d4t1c3']
input_files = list(
    map(
        lambda cw: CellWrapper(cw.info, ResultPaths(cw.data, cw.data)),
        collect_processed_paths_for_infos(
            cell_infos, params.data.spike_features.model_dump()['results']
        )
))

In [9]:
pool_df = collect_pooled_data(input_files)
agg_df = collect_aggregated_data(input_files)

In [10]:
sorted_pool_df = agg_df.sort_values(by='mean')

In [11]:
figure_path = Path(f'{params.output.path}/02_figures')
figure_path.mkdir(parents=True, exist_ok=True)
result_path = Path(params.output.path) / '01_results'
result_path.mkdir(parents=True, exist_ok=True)

n_cells = len(sorted_pool_df)
fig, ax = plt.subplots(
    1, 1, figsize=(20, int(round(0.50 * n_cells)))
)

x_limit = (0, 0.1)
cell_palette = params.palette['cell_type']
for i, (_, cell_row) in enumerate(sorted_pool_df.iterrows()):
    cell_type = cell_row['cell_type']
    cell_color = cell_palette[cell_type]
    cell_props = cell_prop_table[
        (cell_prop_table['name'] == cell_row['cell_name']) &
        (cell_prop_table['group'] == cell_row['cell_type'])
    ].iloc[0, :]
    is_selected = cell_props['selected'] == 1
    bg_color = "#ffff00" if is_selected else "#ababab"
    ax.fill_betweenx(
        y=[i, i + 1],
        x1=x_limit[0],
        x2=x_limit[1],
        color=bg_color,
        alpha=0.3,
        zorder=-1
    )

    ax.fill_betweenx(
        y=[i, i + 1],
        x1=cell_row['q1'],
        x2=cell_row['q3'],
        color=cell_color,
        alpha=0.8,
        zorder=0
    )
    ax.vlines(
        x=cell_row['median'],
        ymin=i,
        ymax=i + 1,
        linewidth=1,
        color='white',
        zorder=1
    )
    ax.vlines(
        x=cell_row['mean'],
        ymin=i,
        ymax=i + 1,
        linewidth=1,
        color='red',
        zorder=1
    )

ax.set_xlim(0, 0.1)
ax.set_ylim(0, len(sorted_pool_df))
ax.set_yticks(np.arange(0.5, len(sorted_pool_df) + 0.5, 1))
ax.set_yticklabels(sorted_pool_df['cell_name'].tolist())
fig.savefig(
    f'{figure_path}/nrmsd_boxes.svg', format='svg'
)
plt.close(fig)

sorted_pool_df.to_excel(
    f'{result_path}/aggregated_noise_data.xlsx', index=False
)

In [15]:
n_cells = len(sorted_pool_df)
fig, ax = plt.subplots(
    1, 1, figsize=(20, 10)
)

sns.histplot(
    data=pool_df,
    x='nrmsd',
    bins=np.arange(0, 0.021, 0.00005)
)

fig.savefig(
    f'{figure_path}/nrmsd_pool_hist.svg', format='svg'
)
plt.close(fig)

sorted_pool_df.to_excel(
    f'{result_path}/pooled_noise_data.xlsx', index=False
)